# Egyptian Premier League — schedule optimizer (Google Colab)

This notebook runs the same pipeline as `python -m schedule_optimizer` on your machine: load workbooks → build model → CP-SAT → write files under `output/`.

**Inputs:** the loader expects Excel/CSV under `data/` and `data/Sources/` (see PRD). If your GitHub clone does not include large `.xlsx` files, copy them from Google Drive or upload them into the repo’s `data/` tree before running the solve cell.

## 1) Configuration

- `REPO_URL`: Git remote (change if you use a fork).
- `DATA_FROM_DRIVE`: if set to a **folder on Drive** that mirrors `data/` (contains `Data_Model.xlsx`, `expanded_calendar.xlsx`, and `Sources/…`), those files are copied into the clone after checkout. Leave `None` if everything is already in the repo.

In [ ]:
import os
from pathlib import Path
from typing import Optional

REPO_URL = "https://github.com/ghassanelgendy/egyptian-premier-league-schedule-optimizer.git"
BRANCH = "main"
WORKDIR = Path("/content")
REPO_DIR = WORKDIR / "egyptian-premier-league-schedule-optimizer"

# Example: Path("/content/drive/MyDrive/epl-optimizer-data") with subfolders Data_Model.xlsx, expanded_calendar.xlsx, Sources/
DATA_FROM_DRIVE: Optional[Path] = None

# Optional solver tuning (see pipeline / env usage in repo)
os.environ.setdefault("EPL_CAF_BUFFER_DAYS", "1")
# os.environ["EPL_PHASE2_TIME_LIMIT_S"] = "300"
# os.environ["EPL_DRR_TRIES"] = "24"

## 2) Clone repository

In [ ]:
import shutil
import subprocess

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.check_call(
    ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
    cwd=str(WORKDIR),
)
print("Cloned to", REPO_DIR)

## 3) Install dependencies (`requirements.txt`)

In [ ]:
subprocess.check_call(
    ["pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")],
)

## 4) (Optional) Copy `data/` from Google Drive

Run the next cell only if `DATA_FROM_DRIVE` is set. It mounts Drive and copies files into `REPO_DIR / "data"`.

In [ ]:
if DATA_FROM_DRIVE is not None:
    from google.colab import drive

    drive.mount("/content/drive")
    src = Path(DATA_FROM_DRIVE)
    if not src.is_dir():
        raise FileNotFoundError(src)
    dest = REPO_DIR / "data"
    if dest.exists():
        shutil.rmtree(dest)
    shutil.copytree(src, dest)
    print("Copied data from", src, "->", dest)
else:
    print("DATA_FROM_DRIVE is None — using data/ from the clone only.")

## 5) Verify required input paths

Stops with a clear error if a workbook is missing.

In [ ]:
data_root = REPO_DIR / "data"
sources = data_root / "Sources"
required = [
    data_root / "Data_Model.xlsx",
    data_root / "expanded_calendar.xlsx",
    sources / "expanded_calendar.csv",
    sources / "calendar.xlsx",
    sources / "teams_data.xlsx",
    sources / "stadiums.xlsx",
    sources / "security matrix.xlsx",
    sources / "Stadium_Distance_Matrix.xlsx",
    sources / "Stadium_Distances_Columns.xlsx",
    sources / "FIFA_Days_UPDATED.xlsx",
    sources / "FIFA Days.xlsx",
    sources / "CAF CL.xlsx",
    sources / "CAF CC.xlsx",
    sources / "cont_blockers_table.xlsx",
    sources / "cont_blockers_csv.csv",
]
missing = [p for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError(
        "Missing input files:\n" + "\n".join(str(p) for p in missing)
    )
print("All required inputs present.")

## 6) Run optimization

In [ ]:
import os
import sys

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from schedule_optimizer.pipeline import run_optimization

result = run_optimization(write_outputs=True)
print(result.message)
print("exit_code", result.exit_code, "solver", result.solver_status, "wall_s", round(result.wall_time_s, 2))
if result.schedule_df is not None:
    display(result.schedule_df.head(15))

## 7) Download `output/` as a zip

In [ ]:
import shutil
from pathlib import Path

from google.colab import files

out_zip = WORKDIR / "epl_optimizer_output.zip"
if out_zip.exists():
    out_zip.unlink()
shutil.make_archive(str(WORKDIR / "epl_optimizer_output"), "zip", REPO_DIR / "output")
files.download(str(out_zip))